# Neural Amp Modeler - Batch Model Training Notebook
🔶**Before you run**🔶

Make sure to get a GPU! (From the upper-left menu, click Runtime->Change runtime type->Select "GPU" from the "Hardware accelerator dropdown menu)

This notebook trains **all of your captures in one run**. Put `input.wav`, every reamped `output_*.wav`, and a `captures.json` describing each one into a `NAM_uploads` folder in your Google Drive (recommended — avoids slow browser uploads, see Step 1), then run the single training cell below. Each finished model is automatically named and uploaded to your Google Drive at `NAM/[gear_model]/[name].nam` — no need to babysit it between captures, however long it takes.

## Step 1: Prepare your files
* **Download the reamp signal.** Here: [input.wav](https://drive.google.com/file/d/1KbaS4oXXNEuh2aCPLwKrPdf5KFOjda8G/view?usp=drive_link).
* **Reamp your gear.** For each piece of gear or tone you want to model, reamp it using `input.wav` and save the result as its own file, e.g. `output_be100_lead.wav`, `output_jcm800_clean.wav`, etc. *Note: Use 48kHz, 24-bit, mono.* For other sample rates, use [the CLI trainer](https://github.com/sdatkinson/neural-amp-modeler).
* **Create your `captures.json`.** Copy [`captures.example.json`](captures.example.json) from this repo, rename it to `captures.json`, and add one entry per capture. Each entry needs `output` (the reamp filename), `name`, and `gear_model`; everything else (your name, gear make, tone type, epochs, latency, reamp levels, etc.) can be set once in `defaults` and overridden per-capture if needed.
* **Get everything into Google Drive (recommended).** Create a folder named **`NAM_uploads`** at the top level of `MyDrive`, and put `input.wav`, every `output_*.wav` file, and `captures.json` in it. The easiest way for a big batch is the **Google Drive desktop app** — drop the files into the locally-synced `NAM_uploads` folder and they'll upload in the background at your full internet speed (Colab's browser upload widget is much slower for many large files, especially in Firefox).
  * *Quick one-off test:* instead of using Drive, you can upload these files directly into the Colab file browser (folder icon ⬅) at the `/content` root — the notebook falls back to that if `MyDrive/NAM_uploads/captures.json` isn't found.

## Step 2: Train everything!
Run the cell below. It will:
1. Install NAM (first run only).
2. Ask you to authorize access to your Google Drive (one click — needed to read your files from `NAM_uploads` and to upload your finished models).
3. Train a model for every entry in `captures.json`, one after another.
4. Save each finished model to `MyDrive/NAM/[gear_model]/[name].nam`, along with a `[name].png` comparison plot and a `_batch_results_*.json` summary of the whole run.

🕙**This can take a long time for many captures — that's expected.** Each capture trains independently, so a problem with one (bad file, failed data checks, etc.) is recorded in the results summary and the batch just moves on to the next.🕙

In [ ]:
# Hi there! This is the code that makes this notebook work. Feel free to mess around
# with it :)

try:
    import nam
except ImportError as e:
    print("Installing NAM into Colab. This should take a couple seconds.")
    # Check what we're starting with (Issue 399)
    !if [ ! -d logs ]; then mkdir logs; fi
    !pip list > logs/packages.log
    !pip install neural-amp-modeler==0.13 >& logs/install.log
    # Hint: use the next line instead for the very latest!
    # !pip install git+https://github.com/sdatkinson/neural-amp-modeler.git@main

import json
import re
import shutil
import traceback
from datetime import datetime
from pathlib import Path
from typing import Optional

from nam.train.core import train
from nam.train.metadata import TRAINING_KEY
from nam.models.metadata import GearType, ToneType, UserMetadata
from google.colab import drive, runtime

%load_ext tensorboard

CONTENT_DIR = Path("/content")
WORK_DIR = CONTENT_DIR / "batch_work"
SEED = 0

# DRY: "(optional)" used below in _parse_level_dbu
reamp_default_value = "(optional)"

# A few helper functions to make sense of the values provided in captures.json.
def _verbose_enum(E, val):
    try:
        return E(val)
    except ValueError as e:
        raise ValueError(
            str(e)
            + "\nValid choices are: "
            + ", ".join(list(x.value for x in E))
        )

def _parse_latency(ls) -> Optional[int]:
    ls = str(ls)
    if ls.lower() == "auto":
        return None
    try:
        return int(ls)
    except ValueError as e:
        raise ValueError(
            f"Invalid value for latency {ls} was given. Either use 'auto' or provide "
            f"the reamp latency, in samples.\nOriginal exception:\n\n{e}"
        )

def _parse_level_dbu(value) -> Optional[float]:
    value = "" if value is None else str(value)
    if value in {"", reamp_default_value}:
        return None
    try:
        return float(value)
    except ValueError as e:
        raise ValueError(
            f"Error parsing provided calibration level '{value}'. Either specify a "
            f"number or leave blank.\n\nOriginal exception: {e}"
        )

def _safe_filename(s: str) -> str:
    """Make a string safe to use as a file or folder name."""
    return re.sub(r'[<>:"/\\|?*\n\r\t]', "_", str(s)).strip()

def _unique_basename(folder: Path, name: str) -> str:
    """`name`, or `name (2)`, `name (3)`, ... if `name.nam` already exists in `folder`."""
    base = _safe_filename(name)
    if not (folder / f"{base}.nam").exists():
        return base
    i = 2
    while (folder / f"{base} ({i}).nam").exists():
        i += 1
    return f"{base} ({i})"

def _build_user_metadata(cfg: dict) -> UserMetadata:
    return UserMetadata(
        name=cfg.get("name"),
        modeled_by=cfg.get("modeled_by"),
        gear_make=cfg.get("gear_make"),
        gear_model=cfg.get("gear_model"),
        gear_type=_verbose_enum(GearType, cfg["gear_type"].lower()) if cfg.get("gear_type") else None,
        tone_type=_verbose_enum(ToneType, cfg["tone_type"].lower()) if cfg.get("tone_type") else None,
        input_level_dbu=_parse_level_dbu(cfg.get("reamp_send_level")),
        output_level_dbu=_parse_level_dbu(cfg.get("reamp_return_level")),
    )

# ---- Mount Google Drive (you'll be asked to authorize this once) ----
drive.mount(str(CONTENT_DIR / "drive"))
MYDRIVE = CONTENT_DIR / "drive" / "MyDrive"

# ---- Find your source files: prefer MyDrive/NAM_uploads, fall back to /content ----
UPLOADS_DIR = MYDRIVE / "NAM_uploads"
if (UPLOADS_DIR / "captures.json").exists():
    SOURCE_DIR = UPLOADS_DIR
else:
    SOURCE_DIR = CONTENT_DIR
print(f"Reading capture files from: {SOURCE_DIR}")

INPUT_FILE = SOURCE_DIR / "input.wav"
MANIFEST_FILE = SOURCE_DIR / "captures.json"

# ---- Load the batch manifest ----
if not MANIFEST_FILE.exists():
    raise FileNotFoundError(
        f"captures.json not found in {SOURCE_DIR}. Either put it (with input.wav and "
        f"your output_*.wav files) in MyDrive/NAM_uploads, or upload it directly into "
        f"/content. See captures.example.json in the repo for the format."
    )
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"input.wav not found in {SOURCE_DIR}.")

manifest = json.loads(MANIFEST_FILE.read_text())
defaults = manifest.get("defaults", {})
captures = manifest["captures"]

drive_root = MYDRIVE / manifest.get("drive_folder", "NAM")
drive_root.mkdir(parents=True, exist_ok=True)

%tensorboard --logdir /content/batch_work

# ---- Train each capture, one after another ----
results = []
for i, capture in enumerate(captures, 1):
    cfg = {**defaults, **capture}
    name = cfg.get("name")
    gear_model = cfg.get("gear_model")
    output_name = cfg.get("output")
    print(f"\n{'=' * 60}\n[{i}/{len(captures)}] {gear_model} / {name}\n{'=' * 60}")

    try:
        if not name or not gear_model or not output_name:
            raise ValueError("Each capture needs 'output', 'name', and 'gear_model'.")

        output_path = SOURCE_DIR / output_name
        if not output_path.exists():
            raise FileNotFoundError(f"Reamp file {output_name} not found in {SOURCE_DIR}.")

        user_metadata = _build_user_metadata(cfg)

        work_dir = WORK_DIR / f"{i:03d}_{_safe_filename(name)}"
        work_dir.mkdir(parents=True, exist_ok=True)
        local_basename = "model"

        train_output = train(
            input_path=str(INPUT_FILE),
            output_path=str(output_path),
            train_path=str(work_dir),
            epochs=int(cfg.get("epochs", 100)),
            latency=_parse_latency(cfg.get("latency", "auto")),
            seed=SEED,
            save_plot=True,
            silent=True,
            modelname=local_basename,
            ignore_checks=bool(cfg.get("ignore_checks", False)),
            local=False,
            user_metadata=user_metadata,
        )

        if train_output is None or train_output.model is None:
            raise RuntimeError(
                "Training did not produce a model (data checks may have failed; "
                "set \"ignore_checks\": true in captures.json to override)."
            )

        model = train_output.model
        training_metadata = train_output.metadata
        esr = getattr(training_metadata, "validation_esr", None)

        model.net.export(
            work_dir,
            basename=local_basename,
            user_metadata=user_metadata,
            other_metadata={TRAINING_KEY: training_metadata.model_dump()},
        )

        dest_dir = drive_root / _safe_filename(gear_model)
        dest_dir.mkdir(parents=True, exist_ok=True)
        final_basename = _unique_basename(dest_dir, name)

        shutil.copy2(work_dir / f"{local_basename}.nam", dest_dir / f"{final_basename}.nam")
        plot_path = work_dir / f"{local_basename}.png"
        if plot_path.exists():
            shutil.copy2(plot_path, dest_dir / f"{final_basename}.png")

        print(f"Done. ESR={esr}. Saved to {dest_dir / f'{final_basename}.nam'}")
        results.append({
            "name": name,
            "gear_model": gear_model,
            "output": output_name,
            "status": "ok",
            "esr": esr,
            "nam_path": str(dest_dir / f"{final_basename}.nam"),
        })
    except Exception as e:
        traceback.print_exc()
        results.append({
            "name": name,
            "gear_model": gear_model,
            "output": output_name,
            "status": "error",
            "error": str(e),
        })
        print(f"FAILED: {e}")

# ---- Save a summary of the whole batch to Drive ----
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
summary_path = drive_root / f"_batch_results_{stamp}.json"
summary_path.write_text(json.dumps(results, indent=2))

num_ok = sum(r["status"] == "ok" for r in results)
print(f"\n{'=' * 60}\nBatch complete: {num_ok}/{len(results)} succeeded.")
print(f"Summary saved to {summary_path}")
for r in results:
    if r["status"] == "ok":
        print(f"  OK    {r['gear_model']}/{r['name']}  (ESR={r['esr']})")
    else:
        print(f"  ERROR {r.get('gear_model')}/{r.get('name')}  ({r['error']})")

# ---- Disconnect the runtime now that everything is in Drive ----
print("\nAll done — disconnecting the runtime.")
runtime.unassign()

## Step 3: Check your results
We're done! Open Google Drive and look in `MyDrive/NAM/`. Each gear model has its own folder containing:
* **`[name].nam`** — the trained model, ready to load in your NAM plugin.
* **`[name].png`** — a plot comparing the model's output to your reamped target (lower ESR is better — have a look to make sure it looks good!).
* **`_batch_results_*.json`** — one per run, summarizing every capture (ESR and pass/fail status).

If a capture failed, check its error message in the results JSON (or the cell output above), fix the issue (e.g. re-export the reamp, tweak `captures.json`), and re-run. Existing `.nam` files in Drive are never overwritten — a re-run that produces the same `name` gets a `(2)`, `(3)`, etc. suffix instead.

# 🎸 **ENJOY!** 🎸

## Troubleshooting: "No latency provided and cannot automatically analyze the latency"

NAM normally finds your reamp's round-trip latency automatically by looking for a
response in your output recording to the calibration "blips" that `input.wav` sends at
roughly 0:10.5 and 0:11.5. If it can't find one, every capture made with the same
setup will fail with this error (since they all share the same latency).

Run the diagnostic cell below (no GPU/training needed, just `input.wav` and one
`output_*.wav` available, either in `MyDrive/NAM_uploads` or uploaded to `/content`).
It plots your input and output around those blip times so you can read off the latency
— the number of samples by which the output lags the input — and shows the levels NAM
compared against its detection threshold.

Once you have a number, set it in `captures.json`:

```jsonc
"defaults": {
  ...,
  "latency": 240   // an integer, in samples — same for every capture from this setup
}
```

Then re-run Step 2. With an explicit `latency`, NAM skips automatic detection entirely.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from nam.data import wav_to_np
from google.colab import drive

CONTENT_DIR = Path("/content")

# ---- Mount Google Drive and find your source files (same logic as Step 2) ----
drive.mount(str(CONTENT_DIR / "drive"))
MYDRIVE = CONTENT_DIR / "drive" / "MyDrive"
UPLOADS_DIR = MYDRIVE / "NAM_uploads"
SOURCE_DIR = UPLOADS_DIR if (UPLOADS_DIR / "captures.json").exists() else CONTENT_DIR
print(f"Reading capture files from: {SOURCE_DIR}")

# Pick the first capture's output file to inspect (change the index if you'd like a
# different one).
manifest = json.loads((SOURCE_DIR / "captures.json").read_text())
output_name = manifest["captures"][0]["output"]

x = wav_to_np(str(SOURCE_DIR / "input.wav"))
y = wav_to_np(str(SOURCE_DIR / output_name))

print(f"Inspecting: {output_name}")
print(f"input.wav length:      {len(x)} samples")
print(f"{output_name} length: {len(y)} samples")

# Where input.wav's standard (v3) calibration blips are, and the quiet stretch just
# before them that NAM uses to measure the background noise level.
BLIP_SAMPLES = (504_000, 552_000)
NOISE_INTERVAL = (492_000, 498_000)
WINDOW = 2_000  # samples shown before/after each blip in the plots

background_level = float(abs(y[NOISE_INTERVAL[0]:NOISE_INTERVAL[1]]).max())
abs_threshold, rel_threshold = 0.0003, 0.001
trigger_threshold = max(background_level + abs_threshold, (1 + rel_threshold) * background_level)
print(f"\nBackground level near the blips: {background_level:.6f}")
print(f"NAM's detection threshold:        {trigger_threshold:.6f}")

for blip in BLIP_SAMPLES:
    lo, hi = blip - WINDOW, blip + WINDOW
    response_peak = float(abs(y[lo:hi]).max())
    detected = "YES" if response_peak > trigger_threshold else "no"
    print(
        f"\nBlip at sample {blip} (~{blip / 48000:.2f}s): "
        f"peak output level in this window = {response_peak:.6f} "
        f"(above threshold? {detected})"
    )
    fig, axs = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    axs[0].plot(range(lo, hi), x[lo:hi])
    axs[0].set_title("input.wav (look for the spike)")
    axs[1].plot(range(lo, hi), y[lo:hi])
    axs[1].set_title(f"{output_name} (look for the response)")
    axs[1].set_xlabel("Sample")
    plt.tight_layout()
    plt.show()

print(
    "\nHow to read this:\n"
    "- If the bottom plot shows a clear response near the spike, the 'latency' is\n"
    "  (sample where the response starts) - (sample where the input spike is), i.e.\n"
    f"  response_sample - {BLIP_SAMPLES[0]} (or - {BLIP_SAMPLES[1]} for the second blip).\n"
    "  Set that as \"latency\" (an integer) in captures.json's \"defaults\".\n"
    "- If 'peak output level' is near zero and below the threshold for BOTH blips,\n"
    "  there's no usable response here at all -- the recording may be too quiet\n"
    "  (check input gain / levels), silent at this point, or not time-aligned with\n"
    "  input.wav (e.g. it was trimmed, or input.wav isn't the unmodified file from\n"
    "  the README link)."
)